In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error, r2_score
import statsmodels.api as sm
from scipy.interpolate import interp1d

# Načtení dat (Zkontroluj, jestli se sloupce opravdu jmenují 'X' a 'Y')
df1 = pd.read_csv('Synthetic dataset 1.csv')
X = df1['X'].values
y = df1['Y'].values

# 1. Rozdělení na trénovací a testovací množinu (70/30)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Pro LOESS je lepší mít trénovací data seřazená podle X
sort_idx = np.argsort(X_train)
X_train_sorted = X_train[sort_idx]
y_train_sorted = y_train[sort_idx]

In [ ]:
# 2. Natrénování LOESS modelu na trénovacích datech
# frac = 0.3 určuje flexibilitu (jak velký úsek dat se bere pro lokální vyhlazení)
loess_fit = sm.nonparametric.lowess(y_train_sorted, X_train_sorted, frac=0.3)

# Vytvoření interpolační funkce pro predikci na testovacích datech
loess_interp = interp1d(loess_fit[:, 0], loess_fit[:, 1], bounds_error=False, fill_value="extrapolate")
y_pred_loess = loess_interp(X_test)

# 3. Výběr stupně polynomu pomocí RMSE na trénovacích datech
best_d = 1
best_rmse = float('inf')
best_poly_model = None
best_poly_features = None

for d in range(1, 8):
    poly = PolynomialFeatures(degree=d)
    X_train_poly = poly.fit_transform(X_train.reshape(-1, 1))

    model = LinearRegression().fit(X_train_poly, y_train)
    y_train_pred = model.predict(X_train_poly)

    rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    if rmse < best_rmse:
        best_rmse = rmse
        best_d = d
        best_poly_model = model
        best_poly_features = poly

print(f"Nejlepší stupeň polynomu vybraný podle trénovacího RMSE je: d = {best_d}")

# Predikce polynomu na testovací sadě
X_test_poly = best_poly_features.transform(X_test.reshape(-1, 1))
y_pred_poly = best_poly_model.predict(X_test_poly)

In [ ]:
# 4. Vyhodnocení na testovací množině
results = {
    'Model': ['LOESS', f'Polynom (d={best_d})'],
    'RMSE': [
        np.sqrt(mean_squared_error(y_test, y_pred_loess)),
        np.sqrt(mean_squared_error(y_test, y_pred_poly))
    ],
    'R2': [
        r2_score(y_test, y_pred_loess),
        r2_score(y_test, y_pred_poly)
    ]
}
print(pd.DataFrame(results).set_index('Model'))

# 5. Vizuální porovnání (Skutečné vs. Predikované hodnoty)
plt.figure(figsize=(12, 5))

# Graf 1: Aktuální vs Predikované pro LOESS
plt.subplot(1, 2, 1)
plt.scatter(y_test, y_pred_loess, alpha=0.6, color='blue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2) # Ideální přímka
plt.title('LOESS: Skutečné vs. Predikované')
plt.xlabel('Skutečné Y')
plt.ylabel('Predikované Y')
plt.grid(True)

# Graf 2: Aktuální vs Predikované pro Polynom
plt.subplot(1, 2, 2)
plt.scatter(y_test, y_pred_poly, alpha=0.6, color='green')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.title(f'Polynom (d={best_d}): Skutečné vs. Predikované')
plt.xlabel('Skutečné Y')
plt.grid(True)

plt.tight_layout()
plt.show()

# Extra: Vykreslení křivek přes původní data pro lepší představu
X_plot = np.linspace(X.min(), X.max(), 200)
plt.figure(figsize=(8, 5))
plt.scatter(X_test, y_test, color='gray', label='Testovací data', alpha=0.5)
plt.plot(X_plot, loess_interp(X_plot), color='blue', label='LOESS křivka', lw=2)
plt.plot(X_plot, best_poly_model.predict(best_poly_features.transform(X_plot.reshape(-1, 1))), color='green', label=f'Polynom d={best_d}', lw=2)
plt.title('Vizuální porovnání modelů na datech')
plt.xlabel('X')
plt.ylabel('Y')
plt.legend()
plt.show()

### 6. Diskuze: LOESS vs. Polynomiální regrese
* **LOESS (Nekparametrický model):** Je velmi flexibilní, protože nepožaduje specifikaci globálního matematického vzorce. Přizpůsobuje se datům lokálně. Výhodou je, že dokáže zachytit i velmi složité a nečekané tvary závislosti. Nevýhodou je, že nemáme k dispozici konkrétní rovnici (parametry), hůře se s ním extrapoluje mimo rozsah dat a je výpočetně náročnější pro velké datasety.
* **Polynomiální regrese (Parametrický model):** Definuje předem globální strukturu modelu (např. rovnici 3. stupně). Výsledkem jsou konkrétní parametry ($\beta_0, \beta_1, ...$), které tvoří rovnici. Je méně flexibilní než LOESS a při volbě příliš vysokého stupně je extrémně náchylná k přeučení (křivka začne divoce oscilovat). Dobře se ale interpretuje a snadno extrapoluje.

In [ ]:
import seaborn as sns

# Načtení dat (Zkontroluj, jak se přesně jmenuje cílová proměnná! Tady předpokládám 'Price')
df_house = pd.read_csv('House price.csv')

# 1. Průzkum dat (EDA)
print("--- Základní informace o datech ---")
print(df_house.info())
print("\n--- Chybějící hodnoty ---")
print(df_house.isnull().sum())

# Případné odstranění/imputace chybějících hodnot (zde pro jednoduchost mažu řádky s chybějícími daty)
df_house = df_house.dropna()

# Předpokládám, že cílová proměnná je 'Price'. Pokud ne, UPRAV TENTO ŘÁDEK!
target_col = 'Price'

# 2. Diagnostika prediktorů - Korelační matice
plt.figure(figsize=(10, 8))
corr_matrix = df_house.corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Korelační matice prediktorů a cílové proměnné')
plt.show()

**Analýza multikolinearity:**
Z korelační matice hledáme vysoké hodnoty (blížící se 1 nebo -1) mezi *prediktory navzájem* (nikoliv mezi prediktorem a cílovou proměnnou, to je naopak žádoucí). Pokud bychom například zjistili, že `Počet_pokojů` a `Obytná_plocha` mají korelaci 0.85, znamená to, že nesou téměř stejnou informaci. To způsobuje multikolinearitu, která ztěžuje interpretaci vlivu jednotlivých proměnných (t-testy koeficientů mohou vyjít nevýznamné, i když celý model funguje dobře).

In [ ]:
# Příprava proměnných
X_multi = df_house.drop(columns=[target_col])
y_multi = df_house[target_col]

# 3. Rozdělení dat: 70% train, 30% test
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_multi, y_multi, test_size=0.3, random_state=42)

# 4. Odhad parametrů modelu (Y = β0 + β1X1 + ... + βpXp + ε)
# statsmodels automaticky nepřidává konstantu (absolutní člen β0), musíme ji přidat ručně
X_train_m_const = sm.add_constant(X_train_m)
X_test_m_const = sm.add_constant(X_test_m)

# Fitování OLS (Ordinary Least Squares) modelu
ols_model = sm.OLS(y_train_m, X_train_m_const).fit()

# 5. Statistická interpretace
# Funkce summary() rovnou vypíše t-testy pro koeficienty, F-test modelu i Adjusted R-squared
print(ols_model.summary())

In [ ]:
# 6. Predikční hodnocení na testovací sadě (test set)
y_pred_multi = ols_model.predict(X_test_m_const)

rmse_multi = np.sqrt(mean_squared_error(y_test_m, y_pred_multi))
r2_multi = r2_score(y_test_m, y_pred_multi)

print("--- Výsledky na testovací množině ---")
print(f"RMSE: {rmse_multi:.2f}")
print(f"R-squared (R2): {r2_multi:.4f}")